# MIDUS2 Data Processing

This module transforms the MIDUS aggregated patient data into a subset of continuous inputs:
- HPA axis (cortisol)
- Autonomic nervous system (HRV)
- Catecholamine secretion (sympathetic activation)

And binary/continuous outcomes:
- Cardiovascular risk, physical health issues, anger, depression, and anxiety

# Imports & Config

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from typing import Optional, List, Dict
import warnings

# --- Configuration & Mappings ---

# Missing value indicators
MISSING_INDICATORS = [
    99999.0, 9999.0, 999.0, 99.0, 99998.0, 9998.0, 998.0, 98.0, 7.0
]

# Column Mappings
BIOMARKER_COLUMN_MAP = {
    'M2ID': 'patient_id',
    'B4VB1LF': 'low_freq_hrv_b1',
    'B4VB2LF': 'low_freq_hrv_b2',
    'B4VB1HF': 'high_freq_hrv_b1',
    'B4VB2HF': 'high_freq_hrv_b2',
    'B4VB1RM': 'rmssd_hrv_b1',
    'B4VB2RM': 'rmssd_hrv_b2',
    'B4VB1HR': 'baseline_hr_b1',
    'B4VB2HR': 'baseline_hr_b2',
    'B4BCLCRE': 'urine_cortisol_adj',
    'B4BEPCRE': 'urine_epinephrine_adj',
    'B4BNOCRE': 'urine_norepinephrine_adj'
}

OUTCOME_COLUMN_MAP = {
    'M2ID': 'patient_id',
    'B4H1A': 'heart_disease',
    'B4H1I': 'diabetes',
    'B4H1F': 'stroke',
    'B4H1B': 'high_bp',
    'B4QTA_AX': 'anxiety',
    'B4QMA_A': 'distress_anxious',
    'B4QCESDI': 'depression_i',
    'B4QCESDPA': 'depression_pa',
    'B4QCESDDA': 'depression_da',
    'B4QCESDSC': 'depression_sc',
    'B4QTA_AG': 'anger',
    'B4HOHLTH': 'phys_hlth_evnts_tot'
}

COVARIATE_COLUMN_MAP = {
    'M2ID': 'patient_id',
    'B1PRAGE_2019': 'age', # updated as of 8-5-2019
    'B1SG8A': 'income',
    'B1PB1': 'education',
    'B1SA37B': 'height_in',
    'B1SA39': 'weight_ib',
    'B1PA38A': 'ever_smoked',
    'B1PA39': 'now_smokes', #1=yes 2=no
    'B4H25': 'phys_active', # any type for 20 minutes or more at least 3 times/week?”
    'B1PF7A': 'race_1'
}

# Utility Methods

In [ ]:
def extract_columns(df: pd.DataFrame, column_map: Dict[str, str]) -> pd.DataFrame:
    """Extract and rename specific columns."""
    return df[list(column_map.keys())].rename(columns=column_map)

In [ ]:
def report_missing_values(df: pd.DataFrame, column_subset: Optional[List[str]] = None, verbose: bool = True):
    """Report missing values based on global MISSING_INDICATORS."""
    cols_to_check = column_subset or df.select_dtypes(include=[np.number]).columns.tolist()

    if verbose:
        print("=" * 40)
        print("MISSING VALUES REPORT")
        for col in cols_to_check:
            if col in df.columns:
                # Check against global constant
                missing_count = df[col].isin(MISSING_INDICATORS).sum()
                if missing_count > 0:
                    print(f"{col}: {missing_count} missing values")

    missing_pct = (df[cols_to_check].isin(MISSING_INDICATORS).sum() / len(df)) * 100
    return missing_pct.sort_values(ascending=False)

In [ ]:
def replace_missing_with_nan(df: pd.DataFrame, column_subset: Optional[List[str]] = None) -> pd.DataFrame:
    """Replace missing indicators with NaN."""
    df_copy = df.copy()
    cols_to_process = column_subset or df_copy.columns.tolist()

    for col in cols_to_process:
        if col in df_copy.columns and col != 'patient_id':
            df_copy[col] = df_copy[col].replace(MISSING_INDICATORS, np.nan)
    return df_copy

In [ ]:
def remove_rows_with_nan(df: pd.DataFrame, subset: Optional[List[str]] = None) -> pd.DataFrame:
    """Remove rows containing NaN values."""
    initial_rows = len(df)
    df_clean = df.dropna(subset=subset).copy()
    removed = initial_rows - len(df_clean)
    print(f"Dropped {removed} rows ({removed/initial_rows*100:.2f}%). Remaining: {len(df_clean)}")
    return df_clean

In [ ]:
def combine_hrv_baselines(df):
    """Complex logic to combine B1 and B2 HRV columns."""
    hrv_pairs = {
        'lf_hrv': ('low_freq_hrv_b1', 'low_freq_hrv_b2'),
        'hf_hrv': ('high_freq_hrv_b1', 'high_freq_hrv_b2'),
        'rmssd': ('rmssd_hrv_b1', 'rmssd_hrv_b2'),
        'hr': ('baseline_hr_b1', 'baseline_hr_b2')
    }

    df_combined = df.copy()

    for new_var, (b1_var, b2_var) in hrv_pairs.items():
        b1_exists = b1_var in df.columns
        b2_exists = b2_var in df.columns

        if not b1_exists and not b2_exists:
            continue

        elif not b1_exists and b2_exists:
            df_combined[new_var] = df[b2_var]

        elif not b2_exists and b1_exists:
            df_combined[new_var] = df[b1_var]

        # Calculate mean across rows ignoring NaNs
        else:
          df_combined[new_var] = df[[b1_var, b2_var]].mean(axis=1)


    return df_combined

## **TODO:** ADD CLEAN COVARIATE METHODS

# Full Execution

## Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load Data

In [ ]:
agg_data = pd.read_csv('/content/drive/MyDrive/MIDUS 2/Data/midus2_agg_data.tsv', sep='\t')

/tmp/ipython-input-2029472511.py:1: DtypeWarning: Columns (2506) have mixed types. Specify dtype option on import or set low_memory=False.
  agg_data = pd.read_csv('/content/drive/MyDrive/MIDUS 2/Data/midus2_agg_data.tsv', sep='\t')


## Process Biomarkers

In [ ]:
print("--- PROCESSING BIOMARKERS ---")

# 1. Extract columns
biomarkers = extract_columns(agg_data, BIOMARKER_COLUMN_MAP)

# 2. Check missing values
report_missing_values(biomarkers)

# 3. Clean Missing Codes
biomarkers = replace_missing_with_nan(biomarkers)

# Debug: Check head
biomarkers.head()

--- PROCESSING BIOMARKERS ---
MISSING VALUES REPORT
low_freq_hrv_b1: 103 missing values
low_freq_hrv_b2: 108 missing values
high_freq_hrv_b1: 167 missing values
high_freq_hrv_b2: 258 missing values
rmssd_hrv_b1: 102 missing values
rmssd_hrv_b2: 107 missing values
baseline_hr_b1: 103 missing values
baseline_hr_b2: 107 missing values
urine_cortisol_adj: 11 missing values
urine_epinephrine_adj: 22 missing values
urine_norepinephrine_adj: 12 missing values


,patient_id,low_freq_hrv_b1,low_freq_hrv_b2,high_freq_hrv_b1,high_freq_hrv_b2,rmssd_hrv_b1,rmssd_hrv_b2,baseline_hr_b1,baseline_hr_b2,urine_cortisol_adj,urine_epinephrine_adj,urine_norepinephrine_adj
0,10002,3226.0,3960.7,NaN,NaN,115.79,95.42,56.2,57.7,3.5,1.698,17.453
1,10005,39.2,65.9,6.21,NaN,5.20,5.21,78.6,77.7,9.1,2.778,43.519
2,10019,58.7,69.6,15.79,13.52,6.90,6.60,77.2,78.4,6.0,0.952,13.492
3,10040,139.1,199.3,66.25,72.70,13.83,14.03,68.3,67.8,19.0,2.208,23.377
4,10047,114.1,292.1,155.80,159.39,17.11,24.12,73.6,75.5,6.5,2.174,32.609


## Combine HRV & remove nans

In [ ]:
print("--- COMBINING HRV BASELINES ---")

# Apply combination logic
biomarkers_combined = combine_hrv_baselines(
    biomarkers
)

# Debug: Check new columns created
new_cols = ['lf_hrv', 'hf_hrv', 'rmssd', 'hr']
print("New columns added:")
print(biomarkers_combined[new_cols].head())
biomarkers_combined.drop(columns=[
    'low_freq_hrv_b1',
    'low_freq_hrv_b2',
    'high_freq_hrv_b1',
    'high_freq_hrv_b2',
    'rmssd_hrv_b1',
    'rmssd_hrv_b2',
    'baseline_hr_b1',
    'baseline_hr_b2'
], inplace=True)


# 4. Remove NaNs
biomarkers_combined = remove_rows_with_nan(biomarkers_combined)

biomarkers_combined.head()

--- COMBINING HRV BASELINES ---
New columns added:
    lf_hrv   hf_hrv    rmssd     hr
0  3593.35      NaN  105.605  56.95
1    52.55    6.210    5.205  78.15
2    64.15   14.655    6.750  77.80
3   169.20   69.475   13.930  68.05
4   203.10  157.595   20.615  74.55
Dropped 196 rows (15.62%). Remaining: 1059


,patient_id,urine_cortisol_adj,urine_epinephrine_adj,urine_norepinephrine_adj,lf_hrv,hf_hrv,rmssd,hr
1,10005,9.1,2.778,43.519,52.55,6.210,5.205,78.15
2,10019,6.0,0.952,13.492,64.15,14.655,6.750,77.80
3,10040,19.0,2.208,23.377,169.20,69.475,13.930,68.05
4,10047,6.5,2.174,32.609,203.10,157.595,20.615,74.55
5,10060,16.0,2.517,32.168,201.00,66.890,19.760,72.35


## Process Outcomes

In [ ]:
print("--- PROCESSING OUTCOMES ---")

# 1. Extract Columns
outcomes = extract_columns(agg_data, OUTCOME_COLUMN_MAP)

# 2. Clean Missing Codes
# (Note: we only process outcome columns, excluding ID)
outcome_cols = [col for col in outcomes.columns if col != 'patient_id']
outcomes = replace_missing_with_nan(outcomes, column_subset=outcome_cols)

# 3. Aggregate Depression Score
depression_cols = ['depression_i', 'depression_pa', 'depression_da', 'depression_sc']
if all(col in outcomes.columns for col in depression_cols):
    outcomes['depression'] = outcomes[depression_cols].sum(axis=1)
    outcomes = outcomes.drop(columns=depression_cols)
    print("Depression score aggregated.")

# 4. Calculate CVD Risk

# This logic checks if ANY of these columns are True/1
cvd_cols = ['heart_disease', 'diabetes', 'stroke', 'high_bp']

# value counts cvd_cols
print("Value counts for CVD columns:")
print(outcomes[cvd_cols].value_counts())

# Ensure columns exist before processing
existing_cvd_cols = [c for c in cvd_cols if c in outcomes.columns]
# create new df column called ["cvd_risk"], value = 0 if all of the columns for current row in in cvd_cols != 1  and 1 if there is at least 1 columns in the current row that has a value of 1
outcomes['cvd_risk'] = (outcomes[existing_cvd_cols] == 1).any(axis=1).astype(int)

# print value counts of true and false
print(outcomes['cvd_risk'].value_counts())

# 5. Remove NaNs
outcomes_clean = remove_rows_with_nan(outcomes)

# 6. Drop original CVD columns (optional, based on your original code)
outcomes_clean = outcomes_clean.drop(columns=existing_cvd_cols, errors='ignore')

# Debug
outcomes_clean.head()

--- PROCESSING OUTCOMES ---
Depression score aggregated.
Value counts for CVD columns:
heart_disease  diabetes  stroke  high_bp
2.0            2.0       2.0     2.0        635
                                 1.0        277
               1.0       2.0     1.0         66
1.0            2.0       2.0     2.0         58
                                 1.0         57
2.0            1.0       2.0     2.0         49
1.0            1.0       2.0     1.0         21
2.0            2.0       1.0     1.0         14
1.0            2.0       1.0     1.0         10
2.0            2.0       1.0     2.0         10
               1.0       1.0     1.0          8
1.0            2.0       1.0     2.0          7
2.0            2.0       2.0     3.0          4
1.0            1.0       1.0     1.0          3
                         2.0     2.0          3
                         1.0     2.0          2
               2.0       2.0     3.0          1
2.0            2.0       3.0     2.0          1
        

,patient_id,anxiety,distress_anxious,anger,phys_hlth_evnts_tot,depression,cvd_risk
0,10002,38.0,16.0,21.0,1,16.0,1
1,10005,21.0,11.0,15.0,0,12.0,1
3,10040,30.0,13.0,20.0,1,10.0,0
4,10047,30.0,16.0,27.0,0,15.0,0
5,10060,25.0,14.0,24.0,1,14.0,1


## Merge Biomarkers and Outputs

In [ ]:
print("--- MERGING DATA ---")

merged_data = pd.merge(biomarkers_combined, outcomes_clean, on='patient_id', how='inner')

print(f"Biomarker rows: {len(biomarkers_combined)}")
print(f"Outcome rows: {len(outcomes_clean)}")
print(f"Merged rows: {len(merged_data)}")
# print data loss
print(f"Data loss: {(1 - len(merged_data)/len(biomarkers_combined))*100:.2f}%")

--- MERGING DATA ---
Biomarker rows: 1059
Outcome rows: 1216
Merged rows: 1030
Data loss: 2.74%


## Standardization

In [ ]:
print("--- STANDARDIZING ---")

final_data = merged_data.copy()
scaler = StandardScaler()

# Identify columns to standardize (all numeric except ID and booleans)
# You can customize this list manually if needed
cols_to_standardize = [
    col for col in final_data.select_dtypes(include=[np.number]).columns
    if col != 'patient_id' and col != 'cvd_risk'
]

print(f"Standardizing: {cols_to_standardize}")

final_data[cols_to_standardize] = scaler.fit_transform(final_data[cols_to_standardize])

print("Standardization complete.")
final_data.head()

--- STANDARDIZING ---
Standardizing: ['urine_cortisol_adj', 'urine_epinephrine_adj', 'urine_norepinephrine_adj', 'lf_hrv', 'hf_hrv', 'rmssd', 'hr', 'anxiety', 'distress_anxious', 'anger', 'phys_hlth_evnts_tot', 'depression']
Standardization complete.


,patient_id,urine_cortisol_adj,urine_epinephrine_adj,urine_norepinephrine_adj,lf_hrv,hf_hrv,rmssd,hr,anxiety,distress_anxious,anger,phys_hlth_evnts_tot,depression,cvd_risk
0,10005,-0.463886,0.806661,1.378215,-0.609466,-0.543491,-1.090312,0.474548,-1.458380,-1.171240,-1.618791,-0.823184,-0.421513,1
1,10040,0.354727,0.304876,-0.242623,-0.418131,-0.412881,-0.529570,-0.457216,-0.456031,-0.756007,-0.703276,0.000000,-0.801720,0
2,10047,-0.678875,0.274945,0.500281,-0.362526,-0.230958,-0.099936,0.142434,-0.456031,-0.133157,0.578446,-0.823184,0.148798,0
3,10060,0.106662,0.576897,0.464794,-0.365971,-0.418218,-0.154886,-0.060525,-1.012892,-0.548390,0.029136,0.000000,-0.041306,1
4,10061,-0.835983,-0.275258,-1.194429,-0.685491,-0.523011,-0.770256,1.175677,-0.790148,-0.963623,-0.337070,0.823184,-0.231410,1


## Save Results

In [ ]:
# Save
output_path = '/content/drive/MyDrive/MIDUS 2/Data/processed_midus_data.csv'
final_data.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to /content/drive/MyDrive/MIDUS 2/Data/processed_midus_data.csv


## Display Summary

In [ ]:
# Summary for all columns but patient_id
print("--- SUMMARY ---")
final_data.describe()

--- SUMMARY ---


,patient_id,urine_cortisol_adj,urine_epinephrine_adj,urine_norepinephrine_adj,lf_hrv,hf_hrv,rmssd,hr,anxiety,distress_anxious,anger,phys_hlth_evnts_tot,depression,cvd_risk
count,1030.000000,1.030000e+03,1.030000e+03,1.030000e+03,1.030000e+03,1.030000e+03,1.030000e+03,1.030000e+03,1.030000e+03,1.030000e+03,1.030000e+03,1.030000e+03,1.030000e+03,1030.000000
mean,14731.573786,1.724618e-16,-9.140477e-17,-1.957442e-16,-4.139084e-17,4.397777e-17,2.759389e-17,3.725176e-16,-1.207233e-17,-5.786094e-16,2.276496e-16,-3.449237e-18,5.863702e-17,0.442718
std,2666.681654,1.000486e+00,1.000486e+00,1.000486e+00,1.000486e+00,1.000486e+00,1.000486e+00,1.000486e+00,1.000486e+00,1.000486e+00,1.000486e+00,1.000486e+00,1.000486e+00,0.496949
min,10005.000000,-1.166735e+00,-1.382707e+00,-1.842538e+00,-6.911502e-01,-5.525541e-01,-1.255161e+00,-2.639021e+00,-1.569753e+00,-1.171240e+00,-1.618791e+00,-8.231838e-01,-2.702756e+00,0.000000
25%,12382.500000,-6.623377e-01,-6.674427e-01,-6.900391e-01,-5.001226e-01,-4.466740e-01,-6.406744e-01,-7.109146e-01,-7.901476e-01,-7.560065e-01,-7.032756e-01,-8.231838e-01,-4.215132e-01,0.000000
50%,14747.000000,-2.240905e-01,-2.448867e-01,-1.946222e-01,-2.877307e-01,-2.861210e-01,-2.553050e-01,-2.131691e-02,-2.332868e-01,-1.331568e-01,-1.539665e-01,0.000000e+00,-2.314096e-01,0.000000
75%,17069.500000,3.547266e-01,4.281216e-01,4.476737e-01,1.330344e-01,1.926402e-02,3.102570e-01,6.452173e-01,5.463183e-01,4.896929e-01,5.784455e-01,8.231838e-01,3.389012e-01,1.000000
max,19193.000000,1.250989e+01,7.726015e+00,6.519318e+00,1.725450e+01,1.113745e+01,9.379011e+00,3.389770e+00,4.110227e+00,5.264874e+00,5.522227e+00,2.469551e+00,5.281594e+00,1.000000
